# Bridge Element-Level HI Forecasting with Temporal GNN
This notebook builds a static bridge–element graph from the CSV, creates yearly feature snapshots (2020–2024), trains a Temporal GCN+GRU model to predict next-year element HI, and produces 5-year forecasts (2025–2029) with:
- **Autoregressive HI** feedback (`HI_prev`)
- **Scenario-based** evolution of bridge age, traffic, and weather covariates.

Outputs:
- `element_hi_forecast_2025_2029_AR_scenario.csv`


## 0) Setup
Make sure you have `torch`, `torch_geometric`, and their dependencies installed in the same kernel.
If imports fail, install PyTorch first, then install PyG matching your torch version.

In [ ]:
import pandas as pd
import numpy as np

# ---- paths ----
CSV_PATH = "../data/NEW_NBE_and_NBI_FL.csv"  # update if needed

# ---- column names ----
TARGET_COL = "HI"
YEAR_COL = "YEAR"
BRIDGE_ID_COL = "STRUCTURE_NUMBER"
ELEM_NUM_COL = "ELEMENT_NUMBER"
PARENT_ELEM_NUM_COL = "PARENT_ELEMENT_NUMBER"

YEARS = [2020, 2021, 2022, 2023, 2024]


## 1) Graph + snapshot builders
These functions:
- create global node indices for bridge and element nodes
- build static edges (bridge↔element, parent↔child)
- one-hot encode categorical columns
- **coerce to numeric and impute missing/infinite values** (critical to avoid NaNs in GNN)
- build yearly feature matrices `X_t` and labels `y_t`.


In [ ]:
from sklearn.feature_extraction import FeatureHasher
import hashlib

def normalize_bridge_id(x) -> str:
    """Ensure STRUCTURE_NUMBER becomes a stable string (avoid float formatting)."""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except Exception:
        return str(x)

def build_global_node_index(df: pd.DataFrame):
    bridge_ids = df[BRIDGE_ID_COL].map(normalize_bridge_id)
    elem_nums = df[ELEM_NUM_COL].astype("Int64")

    bridge_nodes = set(f"bridge:{bid}" for bid in bridge_ids.dropna().unique())
    elem_nodes = set(
        f"elem:{bid}:{int(en)}"
        for bid, en in zip(bridge_ids, elem_nums)
        if (bid is not None) and (not pd.isna(en))
    )

    all_nodes = sorted(list(bridge_nodes | elem_nodes))
    node2idx = {k: i for i, k in enumerate(all_nodes)}
    return node2idx, all_nodes, elem_nodes, bridge_nodes

def build_static_edges(df: pd.DataFrame, node2idx: dict):
    edges = []
    b_ids = df[BRIDGE_ID_COL].map(normalize_bridge_id).values
    e_nums = df[ELEM_NUM_COL].astype("Int64").values
    p_nums = df[PARENT_ELEM_NUM_COL].astype("Int64").values if PARENT_ELEM_NUM_COL in df.columns else [pd.NA]*len(df)

    for bid, en, pn in zip(b_ids, e_nums, p_nums):
        if bid is None or pd.isna(en):
            continue

        b_key = f"bridge:{bid}"
        e_key = f"elem:{bid}:{int(en)}"

        if b_key in node2idx and e_key in node2idx:
            b_i = node2idx[b_key]
            e_i = node2idx[e_key]
            edges.append((b_i, e_i))
            edges.append((e_i, b_i))

        if (pn is not None) and (not pd.isna(pn)):
            p_key = f"elem:{bid}:{int(pn)}"
            if p_key in node2idx and e_key in node2idx:
                p_i = node2idx[p_key]
                e_i = node2idx[e_key]
                edges.append((p_i, e_i))
                edges.append((e_i, p_i))

    edges = list(set(edges))
    edge_index = np.array(edges, dtype=np.int64).T
    return edge_index

def hash_categorical_features(df: pd.DataFrame, n_features: int = 128):
    """
    Replace one-hot encoding with feature hashing to reduce memory.
    Maps each categorical feature to a fixed number of hash buckets.
    
    Parameters:
    -----------
    n_features : Fixed number of hash dimensions per feature (default 128)
                 Adjust based on dataset size and memory constraints.
    """
    work = df.copy()

    # bool -> int
    bool_cols = [c for c in work.columns if work[c].dtype == bool]
    for c in bool_cols:
        work[c] = work[c].astype(np.int8)

    # Don't hash IDs
    id_like = {
        YEAR_COL, BRIDGE_ID_COL,
        "LATEST_STRUCTURE_NUMBER", "NBI_8 - Structure Number", "NBI_2025 NBI Structure Number"
    }
    obj_cols = [c for c in work.columns if work[c].dtype == "object" and c not in id_like]

    # Drop ID columns early
    drop_cols = [c for c in list(id_like) if c in work.columns]
    work = work.drop(columns=drop_cols, errors="ignore")

    # Hash each categorical column independently
    hash_features = []
    for col in obj_cols:
        col_name = col
        # Convert to string, handle NaN
        col_values = work[col].fillna("__NA__").astype(str).values
        
        # Create hash-based features: hash each value to range [0, n_features)
        hashed = np.zeros((len(col_values), n_features), dtype=np.float32)
        for i, val in enumerate(col_values):
            # Use hash to determine which buckets activate for this value
            h = int(hashlib.md5(f"{col_name}_{val}".encode()).hexdigest(), 16)
            for j in range(n_features):
                # Multiple hash outputs populate different buckets
                bucket = (h + j) % n_features
                hashed[i, bucket] += 1.0 / (n_features // 4)  # Normalize contribution
        
        hash_features.append(hashed)
    
    # Concatenate all hash features
    if hash_features:
        X_hashed = np.concatenate(hash_features, axis=1)
    else:
        X_hashed = np.zeros((len(work), 0), dtype=np.float32)
    
    # Get numeric columns (excluding the ones we already dropped)
    numeric_cols = [c for c in work.select_dtypes(include=[np.number]).columns]
    X_numeric = work[numeric_cols].values.astype(np.float32)
    
    # Combine numeric + hashed features
    work_final = np.concatenate([X_numeric, X_hashed], axis=1)
    work_final = np.nan_to_num(work_final, nan=0.0)
    work_final = np.where(np.isinf(work_final), 0.0, work_final)
    
    feature_names = numeric_cols + [f"hash_{i}" for i in range(X_hashed.shape[1])]
    
    return pd.DataFrame(work_final, columns=feature_names)

def make_feature_frame(df: pd.DataFrame):
    """
    Process features using hashing instead of one-hot encoding.
    Much more memory-efficient for large datasets.
    """
    # Use hashing with 128 buckets per categorical feature
    # Adjust n_features DOWN if still OOM (try 64, 32)
    # or UP if you have room and want more capacity
    return hash_categorical_features(df, n_features=128)

def build_yearly_matrices(df_raw: pd.DataFrame, node2idx: dict):
    df = df_raw.copy()
    df[BRIDGE_ID_COL] = df[BRIDGE_ID_COL].map(normalize_bridge_id)

    feat_df = make_feature_frame(df)

    drop_cols = [
        TARGET_COL, YEAR_COL, BRIDGE_ID_COL,
        "LATEST_STRUCTURE_NUMBER", "NBI_8 - Structure Number", "NBI_2025 NBI Structure Number"
    ]
    drop_cols = [c for c in drop_cols if c in feat_df.columns]
    feat_df_no_leak = feat_df.drop(columns=drop_cols, errors="ignore")

    feature_cols = list(feat_df_no_leak.columns)
    N = len(node2idx)
    F = len(feature_cols)

    # Node type indicator
    node_type = np.zeros((N, 2), dtype=np.float32)  # [is_bridge, is_element]
    for k, i in node2idx.items():
        if k.startswith("bridge:"):
            node_type[i, 0] = 1.0
        else:
            node_type[i, 1] = 1.0

    X_list, y_list, mask_list = [], [], []

    for year in YEARS:
        sub_raw = df[df[YEAR_COL] == year]
        sub_feat = feat_df_no_leak.loc[sub_raw.index]

        X = np.zeros((N, F + 2), dtype=np.float32)
        y = np.full((N,), np.nan, dtype=np.float32)

        # Element node features + labels
        for ridx, row in sub_raw.iterrows():
            bid = row[BRIDGE_ID_COL]
            en = row[ELEM_NUM_COL]
            if (bid is None) or pd.isna(en):
                continue

            e_key = f"elem:{bid}:{int(en)}"
            if e_key not in node2idx:
                continue
            ni = node2idx[e_key]

            X[ni, :F] = sub_feat.loc[ridx].to_numpy(dtype=np.float32, copy=False)
            y[ni] = float(row[TARGET_COL]) if (TARGET_COL in sub_raw.columns and not pd.isna(row[TARGET_COL])) else np.nan

        # Bridge node features: use first row per bridge-year
        first_per_bridge = sub_raw.dropna(subset=[BRIDGE_ID_COL]).groupby(BRIDGE_ID_COL).head(1)
        for ridx, row in first_per_bridge.iterrows():
            bid = row[BRIDGE_ID_COL]
            b_key = f"bridge:{bid}"
            if b_key not in node2idx:
                continue
            bi = node2idx[b_key]
            X[bi, :F] = sub_feat.loc[ridx].to_numpy(dtype=np.float32, copy=False)

        X[:, F:] = node_type
        mask = ~np.isnan(y)

        X_list.append(X)
        y_list.append(y)
        mask_list.append(mask)

    return X_list, y_list, mask_list, feature_cols

## 2) Build graph + snapshots (2020–2024)

In [ ]:
# Load CSV (low_memory=False helps avoid mixed-type parsing issues)
df_raw = pd.read_csv(CSV_PATH, low_memory=False)

node2idx, idx2node, elem_nodes, bridge_nodes = build_global_node_index(df_raw)
edge_index = build_static_edges(df_raw, node2idx)

X_list, y_list, mask_list, feature_cols = build_yearly_matrices(df_raw, node2idx)

print("Nodes:", len(node2idx), "Edges:", edge_index.shape[1])
print("Snapshots:", len(X_list), "X0:", X_list[0].shape, "labels0:", int(mask_list[0].sum()))

# quick NaN/inf sanity check
for t, year in enumerate(YEARS):
    X = X_list[t]
    print(f"Year {year}: X NaNs={np.isnan(X).sum()} X infs={np.isinf(X).sum()} labeled={int(mask_list[t].sum())}")


## 3) Add autoregressive feature `HI_prev`
We append one feature column to each snapshot representing last-known HI (normalized to 0..1).
During forecasting, we overwrite this feature each year with the model’s own prediction.

In [ ]:
def add_hi_prev_feature(X_list, y_list):
    X_ar = []
    for t in range(len(X_list)):
        X = X_list[t]
        hi_prev = np.nan_to_num(y_list[t].copy(), nan=0.0).astype(np.float32)
        hi_prev_norm = (hi_prev / 100.0).astype(np.float32)
        X_new = np.concatenate([X, hi_prev_norm[:, None]], axis=1)
        X_ar.append(X_new)
    return X_ar

X_list_ar = add_hi_prev_feature(X_list, y_list)
print("Feature dim:", X_list[0].shape[1], "->", X_list_ar[0].shape[1])


## 4) Temporal GNN model (GCN + GRU) with sigmoid output
We predict normalized HI in 0..1 with a sigmoid, then convert back to 0..100 for reporting.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class TemporalGCN_GRU_DeltaMonotonic(nn.Module):
    """
    Temporal GCN + GRU with dropout layers retained for Monte Carlo inference.
    The model predicts next-year normalized HI and normalized delta.
    """
    def __init__(self, in_dim: int, gcn_hidden: int = 128, gru_hidden: int = 128,
                 max_drop: float = 0.15, dropout: float = 0.20):
        super().__init__()
        self.gcn1 = GCNConv(in_dim, gcn_hidden)
        self.gcn2 = GCNConv(gcn_hidden, gcn_hidden)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(input_size=gcn_hidden, hidden_size=gru_hidden, batch_first=False)
        self.post_gru_dropout = nn.Dropout(dropout)
        self.delta_head = nn.Linear(gru_hidden, 1)
        self.max_drop = max_drop

    def forward(self, X_seq, edge_index):
        embeddings = []
        for X in X_seq:
            h = F.relu(self.gcn1(X, edge_index))
            h = self.dropout(h)
            h = F.relu(self.gcn2(h, edge_index))
            h = self.dropout(h)
            embeddings.append(h)

        H = torch.stack(embeddings, dim=0)
        out, _ = self.gru(H)
        out = self.post_gru_dropout(out)

        raw = self.delta_head(out).squeeze(-1)

        hi_prev_seq = torch.stack([X[:, -1] for X in X_seq], dim=0)
        delta = -torch.sigmoid(raw) * self.max_drop * hi_prev_seq
        hi_next = torch.clamp(hi_prev_seq + delta, 0.0, 1.0)
        return hi_next[:-1], delta[:-1]


## 5) Train/test split (temporal)
- Train on: 2020→2021, 2021→2022, 2022→2023
- Test on:  2023→2024


In [ ]:
def train_delta_model(model, X_list, y_list, mask_list, edge_index, lr=1e-3, epochs=120,
                      delta_loss_weight=0.3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)
    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list]

    # normalize targets to 0..1
    y_seq = [torch.tensor(np.nan_to_num(y, nan=np.nan), dtype=torch.float32, device=device) / 100.0
             for y in y_list]
    m_seq = [torch.tensor(m, dtype=torch.bool, device=device) for m in mask_list]

    train_ts = [0, 1, 2]   # 2020->2021, 2021->2022, 2022->2023
    test_t = 3             # 2023->2024

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        pred_hi_next, pred_delta = model(X_seq, edge_index_t)

        train_losses = []
        for t in train_ts:
            mask = m_seq[t + 1]
            if mask.sum() == 0:
                continue

            y_true = y_seq[t + 1][mask]              # actual HI_next
            y_pred = pred_hi_next[t][mask]           # predicted HI_next

            hi_prev = X_seq[t][:, -1][mask]          # previous HI from input
            true_delta = y_true - hi_prev            # actual year-to-year delta
            pred_delta_t = pred_delta[t][mask]

            hi_loss = F.smooth_l1_loss(y_pred, y_true)
            delta_loss = F.smooth_l1_loss(pred_delta_t, true_delta)

            loss_t = hi_loss + delta_loss_weight * delta_loss
            train_losses.append(loss_t)

        loss = torch.stack(train_losses).mean()
        loss.backward()
        opt.step()

        if ep % 20 == 0:
            model.eval()
            with torch.no_grad():
                mask = m_seq[test_t + 1]
                y_true = y_seq[test_t + 1][mask]
                y_pred = pred_hi_next[test_t][mask]

                hi_prev = X_seq[test_t][:, -1][mask]
                true_delta = y_true - hi_prev
                pred_delta_t = pred_delta[test_t][mask]

                hi_loss = F.smooth_l1_loss(y_pred, y_true)
                delta_loss = F.smooth_l1_loss(pred_delta_t, true_delta)
                test_loss = hi_loss + delta_loss_weight * delta_loss

            print(
                f"epoch {ep:04d} | "
                f"train {loss.item():.5f} | "
                f"test {test_loss.item():.5f} | "
                f"test_hi {hi_loss.item():.5f} | "
                f"test_delta {delta_loss.item():.5f}"
            )

    return model

model_delta_mono = TemporalGCN_GRU_DeltaMonotonic(
    in_dim=X_list_ar[0].shape[1],
    gcn_hidden=64,
    gru_hidden=64,
    max_drop=0.15,
    dropout=0.20
)

model_delta_mono = train_delta_model(
    model=model_delta_mono,
    X_list=X_list_ar,
    y_list=y_list,
    mask_list=mask_list,
    edge_index=edge_index,
    lr=1e-3,
    epochs=120,
    delta_loss_weight=0.3
)

torch.save(model_delta_mono.state_dict(), "../models/temporal_gnn_bridge_hi_delta_monotonic.pt")
print("Saved temporal_gnn_bridge_hi_delta_monotonic.pt")

## 6) Evaluate held-out 2023→2024 prediction (MAE/RMSE in HI points)

## 6A) Monte Carlo dropout + baseline evaluation helpers
These helpers keep dropout active at inference, generate uncertainty bands, and compare the GNN against simple baselines.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def enable_mc_dropout(model):
    """Turn on dropout layers while keeping the rest of the model in eval mode."""
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


def weighted_mae(y_true, y_pred, low_threshold=70.0, critical_threshold=50.0,
                 low_weight=2.0, critical_weight=3.0):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return np.nan

    weights = np.ones_like(y_true, dtype=float)
    weights[y_true < low_threshold] = low_weight
    weights[y_true < critical_threshold] = critical_weight
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def prediction_interval_coverage(y_true, mean_pred, std_pred, z=1.96):
    y_true = np.asarray(y_true, dtype=float)
    mean_pred = np.asarray(mean_pred, dtype=float)
    std_pred = np.asarray(std_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(mean_pred) & np.isfinite(std_pred)
    y_true = y_true[valid]
    mean_pred = mean_pred[valid]
    std_pred = std_pred[valid]

    if len(y_true) == 0:
        return np.nan

    lower = mean_pred - z * std_pred
    upper = mean_pred + z * std_pred
    return np.mean((y_true >= lower) & (y_true <= upper))


def collect_train_deltas(y_list, mask_list, node2idx):
    records = []
    elem_lookup = {}

    for node_key, idx in node2idx.items():
        if node_key.startswith("elem:"):
            _, bid, en = node_key.split(":")
            elem_lookup[idx] = {
                "STRUCTURE_NUMBER": bid,
                "ELEMENT_NUMBER": int(en)
            }

    for t in [0, 1, 2]:
        valid_idx = np.where(mask_list[t + 1])[0]
        for idx in valid_idx:
            if idx not in elem_lookup:
                continue

            delta_val = float(y_list[t + 1][idx] - y_list[t][idx])
            if np.isfinite(delta_val):
                records.append({
                    "STRUCTURE_NUMBER": elem_lookup[idx]["STRUCTURE_NUMBER"],
                    "ELEMENT_NUMBER": elem_lookup[idx]["ELEMENT_NUMBER"],
                    "delta": delta_val,
                })

    return pd.DataFrame(records)


def add_element_key(df):
    def map_element(num):
        num = int(num)
        if num in [12]:
            return "deck"
        if num in [31, 32, 33]:
            return "superstructure"
        if num in [41, 42, 43]:
            return "substructure"
        if num in [47]:
            return "bearings"
        if num in [51]:
            return "joints_seals"
        if num in [60, 61]:
            return "railing_parapets"
        if num in [13, 14, 15, 16]:
            return "wearing_surface"
        if num in [71, 72]:
            return "culvert"
        return "superstructure"

    df = df.copy()
    df["element_key"] = df["ELEMENT_NUMBER"].apply(map_element)
    return df


def evaluate_predictions(y_true, y_pred, pred_std=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if pred_std is None:
        valid = np.isfinite(y_true) & np.isfinite(y_pred)
    else:
        pred_std = np.asarray(pred_std, dtype=float)
        valid = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(pred_std)

    dropped = int((~valid).sum())

    y_true_clean = y_true[valid]
    y_pred_clean = y_pred[valid]

    if len(y_true_clean) == 0:
        return {
            "MAE": np.nan,
            "Weighted MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "Dropped Rows": dropped
        }

    metrics = {
        "MAE": mean_absolute_error(y_true_clean, y_pred_clean),
        "Weighted MAE": weighted_mae(y_true_clean, y_pred_clean),
        "RMSE": np.sqrt(mean_squared_error(y_true_clean, y_pred_clean)),
        "R2": r2_score(y_true_clean, y_pred_clean),
        "Dropped Rows": dropped,
    }

    if pred_std is not None:
        pred_std_clean = pred_std[valid]
        metrics["Coverage 95%"] = prediction_interval_coverage(
            y_true_clean, y_pred_clean, pred_std_clean
        )
        metrics["Avg Std"] = np.mean(pred_std_clean)

    return metrics


def monte_carlo_test_prediction(model, X_list_ar, y_list, mask_list, edge_index, n_samples=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)
    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list_ar]
    mask_2024 = mask_list[4]
    true_2024 = y_list[4][mask_2024]

    samples = []
    with torch.no_grad():
        for _ in range(n_samples):
            enable_mc_dropout(model)
            pred_hi_next, _ = model(X_seq, edge_index_t)
            pred_2024 = pred_hi_next[3].detach().cpu().numpy() * 100.0
            samples.append(pred_2024[mask_2024])

    samples = np.asarray(samples, dtype=float)

    # Remove MC runs that contain non-finite predictions
    good_runs = np.all(np.isfinite(samples), axis=1)
    samples = samples[good_runs]

    mean_pred = np.nanmean(samples, axis=0)
    std_pred = np.nanstd(samples, axis=0)

    return true_2024, mean_pred, std_pred, samples

In [ ]:
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_delta_mono = model_delta_mono.to(device)

true_2024, yhat_mean, yhat_std, yhat_samples = monte_carlo_test_prediction(
    model=model_delta_mono,
    X_list_ar=X_list_ar,
    y_list=y_list,
    mask_list=mask_list,
    edge_index=edge_index,
    n_samples=50,
)

hi_prev_2023 = np.asarray(y_list[3][mask_list[4]], dtype=float)
true_2024 = np.asarray(true_2024, dtype=float)
yhat_mean = np.asarray(yhat_mean, dtype=float)
yhat_std = np.asarray(yhat_std, dtype=float)

train_delta_df = collect_train_deltas(y_list, mask_list, node2idx)
train_delta_df = add_element_key(train_delta_df)
train_delta_df = train_delta_df[np.isfinite(train_delta_df["delta"])].copy()

out_eval = add_element_key(pd.DataFrame({
    "ELEMENT_NUMBER": [
        int(k.split(":")[-1])
        for k, idx in node2idx.items()
        if k.startswith("elem:") and mask_list[4][idx]
    ],
}))
test_element_keys = out_eval["element_key"].to_numpy()

avg_delta = float(train_delta_df["delta"].mean()) if len(train_delta_df) else 0.0
if not np.isfinite(avg_delta):
    avg_delta = 0.0

per_elem_delta = (
    train_delta_df.groupby("element_key")["delta"]
    .mean()
    .replace([np.inf, -np.inf], np.nan)
    .to_dict()
)

element_avg_preds = np.array([
    prev + (
        per_elem_delta.get(elem, avg_delta)
        if np.isfinite(per_elem_delta.get(elem, avg_delta))
        else avg_delta
    )
    for prev, elem in zip(hi_prev_2023, test_element_keys)
], dtype=float)

baseline_preds = {
    "Persistence": np.clip(hi_prev_2023, 0.0, 100.0),
    "Global Avg Delta": np.clip(hi_prev_2023 + avg_delta, 0.0, 100.0),
    "Element Avg Delta": np.clip(element_avg_preds, 0.0, 100.0),
    "Temporal GNN + MC Dropout": np.clip(yhat_mean, 0.0, 100.0),
}

for model_name, preds in baseline_preds.items():
    preds = np.asarray(preds, dtype=float)
    print(
        f"{model_name}: "
        f"n={len(preds)}, "
        f"nan_preds={np.isnan(preds).sum()}, "
        f"inf_preds={np.isinf(preds).sum()}"
    )

print(
    f"true_2024: n={len(true_2024)}, "
    f"nan={np.isnan(true_2024).sum()}, "
    f"inf={np.isinf(true_2024).sum()}"
)

summary_rows = []
for model_name, preds in baseline_preds.items():
    metrics = evaluate_predictions(
        true_2024,
        preds,
        pred_std=yhat_std if model_name == "Temporal GNN + MC Dropout" else None
    )
    metrics["Model"] = model_name
    summary_rows.append(metrics)

eval_df = pd.DataFrame(summary_rows)
display(eval_df)

## 7) Scenario-based autoregressive rollout (2025–2029)
We update these covariates each future year:
- **NBI Bridge Age:** +1 each year
- **ADT / traffic:** multiplicative growth (scenario)
- **Weather fields:** linear or % trends (scenario)
and we also update `HI_prev` with the model’s predicted HI.


In [ ]:
from collections import defaultdict

def build_bridge_to_node_indices(node2idx):
    bridge_to_nodes = defaultdict(list)
    for k, i in node2idx.items():
        if k.startswith("bridge:"):
            bid = k.split(":")[1]
            bridge_to_nodes[bid].append(i)
        elif k.startswith("elem:"):
            _, bid, _en = k.split(":")
            bridge_to_nodes[bid].append(i)
    return bridge_to_nodes

bridge_to_nodes = build_bridge_to_node_indices(node2idx)

COVAR_NAMES = [
    "NBI_Bridge Age (yr)",
    "NBI_29 - Average Daily Traffic",
    "NBI_Average Temperature",
    "NBI_Maximum Temperature",
    "NBI_Minimum Temperature",
    "NBI_Number of Freeze-Thaw Cycles",
    "NBI_Number of Days with Measurable Precipitation",
    "NBI_Number of Days with Temperature Below 0?C",
    "NBI_Number of Snowfall Days",
    "NBI_Total Precipitation"
]

def find_feature_index(feature_cols, name):
    return feature_cols.index(name) if name in feature_cols else None

covar_idx = {}
missing = []
for nm in COVAR_NAMES:
    ix = find_feature_index(feature_cols, nm)
    if ix is None:
        missing.append(nm)
    else:
        covar_idx[nm] = ix

print("Found covariates:", list(covar_idx.keys()))
if missing:
    print("Missing (check column spelling):", missing)


In [ ]:
def apply_scenario_updates(X, year_step, covar_idx, bridge_to_nodes, scenario):
    def upd(rows, col, vals):
        X[rows, col] = vals

    adt_growth = scenario.get("adt_growth", 0.02)
    temp_trend = scenario.get("temp_trend_c_per_year", 0.15)
    wind_trend = scenario.get("wind_trend_per_year", 0.0)
    precip_trend_pct = scenario.get("precip_trend_pct_per_year", 0.01)
    freeze_thaw_trend_pct = scenario.get("freeze_thaw_trend_pct_per_year", -0.005)
    below0_trend_pct = scenario.get("below0_days_trend_pct_per_year", -0.01)
    snowfall_trend_pct = scenario.get("snowfall_days_trend_pct_per_year", -0.01)
    precip_days_trend_pct = scenario.get("precip_days_trend_pct_per_year", 0.00)

    for bid, rows in bridge_to_nodes.items():
        rows = np.array(rows, dtype=int)

        if "NBI Bridge Age" in covar_idx:
            c = covar_idx["NBI Bridge Age"]
            upd(rows, c, X[rows, c] + 1.0)

        if "NBI_29 - Average Daily Traffic" in covar_idx:
            c = covar_idx["NBI_29 - Average Daily Traffic"]
            upd(rows, c, X[rows, c] * (1.0 + adt_growth))

        for nm in ["NBI_Avg Daily Max Temp", "NBI_Avg Daily Min Temp",
                   "NBI_Avg Monthly Max Temp", "NBI_Avg Monthly Min Temp",
                   "NBI_Average Temperature", "NBI_Maximum Temperature", "NBI_Minimum Temperature"]:
            if nm in covar_idx:
                c = covar_idx[nm]
                upd(rows, c, X[rows, c] + temp_trend)

        if "NBI_Avg Daily Wind Speed" in covar_idx:
            c = covar_idx["NBI_Avg Daily Wind Speed"]
            upd(rows, c, X[rows, c] + wind_trend)

        for nm, pct in [
            ("NBI_Freeze Thaw Cycles", freeze_thaw_trend_pct),
            ("NBI_Number of Freeze-Thaw Cycles", freeze_thaw_trend_pct),
            ("NBI_Number of Days with Measurable Precipitation", precip_days_trend_pct),
            ("NBI_Number of Days with Temperature Below 0C", below0_trend_pct),
            ("NBI_Number of Days with Temperature Below 0°C", below0_trend_pct),
            ("NBI_Number of Snowfall Days", snowfall_trend_pct),
            ("NBI_Total Precipitation", precip_trend_pct),
        ]:
            if nm in covar_idx:
                c = covar_idx[nm]
                upd(rows, c, X[rows, c] * (1.0 + pct))

    return X


@torch.no_grad()
def rollout_autoregressive_with_covariates_delta(model, X_list_ar, edge_index, years_ahead,
                                                 covar_idx, bridge_to_nodes, scenario):
    device = next(model.parameters()).device
    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)
    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list_ar]
    last_X = X_seq[-1].clone()
    preds_future = []

    for step in range(1, years_ahead + 1):
        model.eval()
        pred_hi_next, _ = model(X_seq, edge_index_t)
        y_hat_next = pred_hi_next[-1]
        preds_future.append((y_hat_next * 100.0).detach().cpu().numpy())

        last_np = last_X.detach().cpu().numpy()
        last_np[:, -1] = y_hat_next.detach().cpu().numpy()
        last_np = apply_scenario_updates(last_np, step, covar_idx, bridge_to_nodes, scenario)
        last_X = torch.tensor(last_np, dtype=torch.float32, device=device)
        X_seq = X_seq[1:] + [last_X.clone()]

    return preds_future


@torch.no_grad()
def rollout_autoregressive_mc_dropout(model, X_list_ar, edge_index, years_ahead,
                                      covar_idx, bridge_to_nodes, scenario,
                                      n_samples=50, risk_threshold=50.0):
    device = next(model.parameters()).device
    edge_index_t = torch.tensor(edge_index, dtype=torch.long, device=device)
    X_seq = [torch.tensor(X, dtype=torch.float32, device=device) for X in X_list_ar]
    last_X = X_seq[-1].clone()

    preds_mean, preds_std, probs_below = [], [], []

    for step in range(1, years_ahead + 1):
        sample_preds = []
        for _ in range(n_samples):
            enable_mc_dropout(model)
            pred_hi_next, _ = model(X_seq, edge_index_t)
            y_hat_next = pred_hi_next[-1]
            sample_preds.append((y_hat_next * 100.0).detach().cpu().numpy())

        sample_preds = np.asarray(sample_preds)
        mean_pred = sample_preds.mean(axis=0)
        std_pred = sample_preds.std(axis=0)
        prob_below = (sample_preds < risk_threshold).mean(axis=0)

        preds_mean.append(mean_pred)
        preds_std.append(std_pred)
        probs_below.append(prob_below)

        last_np = last_X.detach().cpu().numpy()
        last_np[:, -1] = mean_pred / 100.0
        last_np = apply_scenario_updates(last_np, step, covar_idx, bridge_to_nodes, scenario)
        last_X = torch.tensor(last_np, dtype=torch.float32, device=device)
        X_seq = X_seq[1:] + [last_X.clone()]

    return preds_mean, preds_std, probs_below


In [ ]:
scenario = {
    "adt_growth": 0.02,
    "temp_trend_c_per_year": 0.15,
    "precip_trend_pct_per_year": 0.01,
    "freeze_thaw_trend_pct_per_year": -0.005,
    "below0_days_trend_pct_per_year": -0.01,
    "snowfall_trend_pct_per_year": -0.01
}

preds_future_mean, preds_future_std, preds_future_prob_lt50 = rollout_autoregressive_mc_dropout(
    model=model_delta_mono,
    X_list_ar=X_list_ar,
    edge_index=edge_index,
    years_ahead=5,
    covar_idx=covar_idx,
    bridge_to_nodes=bridge_to_nodes,
    scenario=scenario,
    n_samples=50,
    risk_threshold=50.0,
)

elem_rows = []
for node_key, idx in node2idx.items():
    if node_key.startswith("elem:"):
        _, bid, en = node_key.split(":")
        elem_rows.append((idx, bid, int(en)))

out = pd.DataFrame(elem_rows, columns=["node_idx", "STRUCTURE_NUMBER", "ELEMENT_NUMBER"])

for i, year in enumerate([2025, 2026, 2027, 2028, 2029]):
    node_ids = out["node_idx"].to_numpy()
    out[f"HI_pred_{year}"] = preds_future_mean[i][node_ids]
    out[f"HI_std_{year}"] = preds_future_std[i][node_ids]
    out[f"P_HI_lt_50_{year}"] = preds_future_prob_lt50[i][node_ids]

out.to_csv("../results/element_hi_forecast_2025_2029_mc_dropout.csv", index=False)
print("Saved element_hi_forecast_2025_2029_mc_dropout.csv")
out.head()


In [ ]:
import numpy as np
import pandas as pd

# Build node index reverse lookup
idx2node = {idx: node for node, idx in node2idx.items()}

# Pull bridge-level hazard values from the latest available feature snapshot.
# This assumes X_list_ar[-1] is the most recent bridge/node feature matrix.
latest_X = X_list_ar[-1]

def safe_get_feature(row, name, covar_idx, default=0.0):
    if name in covar_idx:
        val = row[covar_idx[name]]
        return float(val) if np.isfinite(val) else default
    return default

bridge_hazards_dict = {}
bridge_meta_dict = {}

for idx, node_name in idx2node.items():
    if not node_name.startswith("bridge:"):
        continue

    bid = node_name.split("bridge:")[1]
    row = latest_X[idx]

    bridge_hazards_dict[bid] = {
        "avg_temperature_c": safe_get_feature(row, "NBI_Average Temperature", covar_idx),
        "max_temperature_c": safe_get_feature(row, "NBI_Maximum Temperature", covar_idx),
        "min_temperature_c": safe_get_feature(row, "NBI_Minimum Temperature", covar_idx),
        "freeze_thaw_cycles": safe_get_feature(row, "NBI_Number of Freeze-Thaw Cycles", covar_idx),
        "precip_days": safe_get_feature(row, "NBI_Number of Days with Measurable Precipitation", covar_idx),
        "below_zero_days": safe_get_feature(row, "NBI_Number of Days with Temperature Below 0°C", covar_idx),
        "snowfall_days": safe_get_feature(row, "NBI_Snowfall Days", covar_idx),
        "wind_speed": safe_get_feature(row, "NBI_Wind Speed", covar_idx, default=0.0),   # optional
        "average_daily_traffic": safe_get_feature(row, "NBI_29 - Average Daily Traffic", covar_idx),
        "truck_percentage": safe_get_feature(row, "NBI_Truck Percentage", covar_idx, default=0.0),  # optional
        "flood_risk_proxy": safe_get_feature(row, "NBI_Flood Risk Proxy", covar_idx, default=0.0),  # optional
    }

    bridge_meta_dict[bid] = {
        "adt": safe_get_feature(row, "NBI_29 - Average Daily Traffic", covar_idx),
        "detour_km": safe_get_feature(row, "NBI_Detour Length (km)", covar_idx, default=0.0),
        "essential": False,
    }

print("bridge_hazards_dict size:", len(bridge_hazards_dict))
print("bridge_meta_dict size:", len(bridge_meta_dict))

In [ ]:
# ── Import criticality pipeline from the dedicated module ──
import sys
import os

# Add the notebooks directory to path so we can import the pipeline module
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd(), "../scripts"))

from bridge_criticality_pipeline_updated import (
    CTIM,
    CriticalityScore,
    LearnedVulnerabilityMatrix,
    BridgePortfolioRanker,
    apply_ctim_to_forecast,
    compute_adt_threshold,
    is_essential_from_data,
    compute_detour_score,
    fit_vulnerability_matrix,
    risk_tier_probabilities,
    dominant_risk_tier,
    weighted_mae,
    regression_metrics,
    BRIDGE_ELEMENTS,
    HAZARDS,
    VULNERABILITY_MATRIX_EXPERT,
    infer_element_key,
    normalize_hazard,
)

print("Criticality pipeline loaded from bridge_criticality_pipeline_updated.py")
print(f"  Elements: {BRIDGE_ELEMENTS}")
print(f"  Hazards : {len(HAZARDS)} total")


## 8) Criticality Pipeline v2: Score Components

The pipeline (imported from ) combines three factors into a 0–100 criticality score:
- **HI Penalty** (40%): Convex penalty for low Health Index — bridges at HI=20 get 3.2× harder penalty than linear
- **CTIM Impact** (40%): Climate-Traffic Impact Matrix — how vulnerable is this element to hazards?
- **Network Significance** (20%): ADT + detour distance + essential route classification

**Key improvement**: Hazard interaction terms capture synergies:
- λ(FT × Truck): Freeze-thaw cycles + heavy trucks → deck spalling
- γ(Precip × Flood): Precipitation + flood-prone area → scour risk


In [ ]:
# Quick demo: See how the pipeline works with sample bridges
print("=" * 80)
print("CRITICALITY PIPELINE v2  —  DEMO: How Climate Affects Bridge Scores")
print("=" * 80)

# Example 1: Coastal bridge in Florida (high flood risk)
coastal_hazards = {
    "avg_temperature_c": 28.0,
    "max_temperature_c": 38.0,
    "min_temperature_c": 12.0,
    "freeze_thaw_cycles": 2.0,
    "precip_days": 130.0,
    "below_zero_days": 0.0,
    "snowfall_days": 0.0,
    "wind_speed": 18.0,
    "average_daily_traffic": 85_000.0,
    "truck_percentage": 0.08,
    "flood_risk_proxy": 0.75,  # High flood risk
}

# Example 2: Rural bridge in cold region (freeze-thaw sensitive)
rural_hazards = {
    "avg_temperature_c": 2.0,
    "max_temperature_c": 22.0,
    "min_temperature_c": -18.0,
    "freeze_thaw_cycles": 50.0,  # High freeze-thaw
    "precip_days": 120.0,
    "below_zero_days": 80.0,
    "snowfall_days": 25.0,
    "wind_speed": 12.0,
    "average_daily_traffic": 5_000.0,
    "truck_percentage": 0.20,  # High truck %
    "flood_risk_proxy": 0.25,
}

# Compute CTIMs
ctim_coastal = CTIM(coastal_hazards)
ctim_rural   = CTIM(rural_hazards)

print("\n--- COASTAL BRIDGE (FL-COASTAL-001) ---")
print(f"Aggregate CTIM impact: {ctim_coastal.aggregate_score:.4f}")
print(f"Top hazards: {ctim_coastal.top_hazards(3)}")
print(f"Most vulnerable elements: {ctim_coastal.top_vulnerable_elements(3)}")
print(f"Interactions: {ctim_coastal.interaction_summary()}")

print("\n--- RURAL BRIDGE (OK-RURAL-042) ---")
print(f"Aggregate CTIM impact: {ctim_rural.aggregate_score:.4f}")
print(f"Top hazards: {ctim_rural.top_hazards(3)}")
print(f"Most vulnerable elements: {ctim_rural.top_vulnerable_elements(3)}")
print(f"Interactions: {ctim_rural.interaction_summary()}")

# Now compute criticality scores at different HI levels
print("\n--- HOW HI AFFECTS CRITICALITY SCORE ---")
print("(Shows: HI=X → score / risk_tier / how climate changes urgency)\n")

for hi_val in [30, 50, 70, 85]:
    cs_coastal = CriticalityScore(
        bridge_id="FL-COASTAL-001", hi=hi_val, ctim=ctim_coastal,
        adt=85_000, detour_score=0.7, essential_route=True, element_key="deck"
    )
    cs_rural = CriticalityScore(
        bridge_id="OK-RURAL-042", hi=hi_val, ctim=ctim_rural,
        adt=5_000, detour_score=0.5, essential_route=False, element_key="deck"
    )
    
    print(f"HI={hi_val}:")
    print(f"  Coastal   → Score: {cs_coastal.score:6.1f}  Tier: {cs_coastal.risk_tier:7s}  "
          f"HI_penalty: {cs_coastal.hi_penalty:.3f}  CTIM: {cs_coastal.ctim_impact:.3f}")
    print(f"  Rural     → Score: {cs_rural.score:6.1f}  Tier: {cs_rural.risk_tier:7s}  "
          f"HI_penalty: {cs_rural.hi_penalty:.3f}  CTIM: {cs_rural.ctim_impact:.3f}")
    print()

print("✓ Demo complete: Pipeline shows how climate + HI + network factors drive urgency")


In [ ]:
# Apply criticality pipeline v2 to forecast
print("=" * 80)
print("APPLYING CRITICALITY PIPELINE v2 TO FORECAST")
print("=" * 80)
print()

# CHECK: Is bridge_meta_dict available?
try:
    # Try to use bridge_meta_dict if it exists
    test = bridge_meta_dict
    print("✓ bridge_meta_dict loaded from earlier cells (cell 22)")
except NameError:
    print("⚠ bridge_meta_dict not found!")
    print("\nREQUIRED: Run cells in this order FIRST:")
    print("  1. Cell 3 (Setup paths & columns)")
    print("  2. Cell 5 (Graph builders)")
    print("  3. Cell 7 (Build graph + snapshots)")
    print("  4. Cell 9 (Add autoregressive feature)")
    print("  5. Cell 11 (Define model)")
    print("  6. Cell 13 (Train model)")
    print("  7. Cell 17 (MC dropout evaluation)")
    print("  8. Cell 19 (Build node indices & covariates)")
    print("  9. Cell 20 (Define scenario updates)")
    print(" 10. Cell 21 (Rollout forecast)")
    print(" 11. Cell 22 (Extract bridge hazards) ← Creates bridge_meta_dict")
    print("\nThen come back and run this cell (31) again.")
    print()
    raise

# Compute ADT threshold for essential bridge classification
adt_series = pd.Series([m.get("adt", 0.0) for m in bridge_meta_dict.values()])
adt_threshold = compute_adt_threshold(adt_series, percentile=75.0)
print(f"ADT threshold (75th percentile): {adt_threshold:,.0f}  → bridges above = ESSENTIAL")
print()

# Apply the pipeline to the forecast
result = apply_ctim_to_forecast(
    forecast_df=out,
    bridge_hazards=bridge_hazards_dict,
    bridge_meta=bridge_meta_dict,
    vuln=None,  # None → use expert priors
    adt_all=adt_series,
    neighbour_table=None,  # Optional: bridge detour alternatives
)

print(f"✓ Criticality scores computed for {len(result)} element-forecasts")
print(f"  Columns: {list(result.columns)}")
print()

In [ ]:
# Analyze how criticality pipeline affects scores
print("=" * 80)
print("CRITICALITY SCORE ANALYSIS — How Pipeline Components Affect Urgency")
print("=" * 80)
print()

print("1. SCORE DISTRIBUTION (0–100 scale):")
print(result["criticality_score"].describe().round(2))
print()

print("2. RISK TIER BREAKDOWN:")
risk_counts = result["risk_tier"].value_counts()
print(risk_counts)
print()

print("3. TOP 15 MOST CRITICAL ELEMENTS (by score):")
top_critical = result.nlargest(15, "criticality_score")[
    ["STRUCTURE_NUMBER", "ELEMENT_NUMBER", "element_key", "HI_pred_2025", 
     "criticality_score", "risk_tier", "hi_penalty", "ctim_impact", "network_significance"]
].copy()
top_critical.columns = ["Bridge_ID", "Elem_Num", "Element", "Forecast_HI", 
                         "Crit_Score", "Tier", "HI_Penalty", "CTIM", "Network_Sig"]
print(top_critical.to_string(index=False))
print()

print("4. IMPACT COMPONENT CORRELATION (shows how each factor contributes):")
component_corr = result[["criticality_score", "hi_penalty", "ctim_impact", "network_significance"]].corr()
print(component_corr.round(3))
print()

print("5. CLIMATE IMPACT DISTRIBUTION:")
print("  CTIM Impact scores (element-level climate-traffic exposure):")
print(result["ctim_impact"].describe().round(4))
print()
print("  Top 5 most climate-vulnerable elements:")
top_climate = result.nlargest(5, "ctim_impact")[["STRUCTURE_NUMBER", "element_key", "ctim_impact"]].drop_duplicates()
print(top_climate.to_string(index=False))
print()

print("6. HI PENALTY (convex vs linear):")
sample_scores = [20, 40, 60, 80]
for hi in sample_scores:
    lin_penalty = 1 - hi/100
    conv_penalty = lin_penalty ** 2
    ratio = conv_penalty / lin_penalty if lin_penalty > 0 else 0
    print(f"  HI={hi:2d}:  linear_penalty={lin_penalty:.3f}  convex_penalty={conv_penalty:.4f}  ratio={ratio:.2f}×")
print()

print("7. ESSENTIAL ROUTE IMPACT:")
essential_count = result["is_essential"].sum()
print(f"  Bridges flagged as essential: {essential_count}")
essential_sample = result[result["is_essential"]][["STRUCTURE_NUMBER", "ctim_impact_bridge", "network_significance"]].drop_duplicates().head()
print(essential_sample.to_string(index=False))
print()

print("✓ Analysis complete!")


In [ ]:
print("Criticality analysis results shape:", result.shape)
print("\nColumns added:")
print(result.columns.tolist())
print("\nSample of element-specific criticality scores:")
print(result[["STRUCTURE_NUMBER", "ELEMENT_NUMBER", "element_key", "criticality_score", "risk_tier", "ctim_impact", "ctim_impact_bridge"]].head(10))
print("\nRisk tier distribution:")
print(result["risk_tier"].value_counts())
print("\nTop 10 most critical bridges (by worst element):")
worst_per_bridge = result.groupby("STRUCTURE_NUMBER")["criticality_score"].max().nlargest(10)
print(worst_per_bridge)


In [ ]:
# Save criticality results
result.to_csv("../results/element_criticality_forecast_2025_2029.csv", index=False)
print("Saved element_criticality_forecast_2025_2029.csv")

# Optional: create a bridge-level summary (worst element per bridge per year)
bridge_criticality = result.groupby("STRUCTURE_NUMBER").agg({
    "criticality_score": "max",
    "risk_tier": lambda x: x.iloc[0],  # Most common
    "ctim_impact": "mean",
    "hi_penalty": "mean",
    "HI_pred_2025": lambda x: result[result["STRUCTURE_NUMBER"] == x.name]["HI_pred_2025"].min() if "HI_pred_2025" in result.columns else 0,
    "HI_pred_2029": lambda x: result[result["STRUCTURE_NUMBER"] == x.name]["HI_pred_2029"].min() if "HI_pred_2029" in result.columns else 0,
}).reset_index()

bridge_criticality.columns = ["STRUCTURE_NUMBER", "criticality_score_max", "risk_tier", "ctim_impact_mean", "hi_penalty_mean", "worst_hi_2025", "worst_hi_2029"]
bridge_criticality = bridge_criticality.sort_values("criticality_score_max", ascending=False)

bridge_criticality.to_csv("../results/bridge_criticality_summary_2025_2029.csv", index=False)
print("Saved bridge_criticality_summary_2025_2029.csv")
print("\nBridge summary (top 10):")
print(bridge_criticality.head(10).to_string())


## 9) Interactive Criticality Dashboard

An interactive HTML dashboard has been created that allows you to explore bridge-specific criticality and deterioration patterns.

### Features:
- **Bridge Selection**: Search and select any bridge from the forecast
- **Multi-Panel Visualizations**:
  - Health Index deterioration trajectory (worst-case vs. bridge mean)
  - Criticality score evolution over 5 years
  - Risk tier probability distribution (pie chart)
  - Top 5 worst elements and their individual trajectories
  - Climate-Traffic Impact Matrix (CTIM) heatmap
  - Element-level detail table with yearly HI, criticality scores, and risk assessments

### How to Use:
1. Copy the dashboard file to your project: `bridge_criticality_interactive_dashboard.html`
2. Place `element_criticality_forecast_2025_2029.csv` in the same directory
3. Open the HTML file in a web browser
4. Use the search or dropdown to select a bridge
5. Click "Load Dashboard" to view all visualizations

The dashboard is fully interactive with hover tooltips, zoom/pan, and animated transitions via Plotly.js.


In [ ]:
# Dashboard instructions and quick stats
import os

# Check if CSV exists and generate info
csv_file = "../results/element_criticality_forecast_2025_2029.csv"
if os.path.exists(csv_file):
    print(f"✓ Found {csv_file}")
    print(f"  File size: {os.path.getsize(csv_file) / 1024 / 1024:.2f} MB")
    
    # Quick stats
    print(f"\nDataset Summary:")
    print(f"  Total element records: {len(result):,}")
    print(f"  Unique bridges: {result['STRUCTURE_NUMBER'].nunique():,}")
    print(f"  Average elements per bridge: {len(result) / result['STRUCTURE_NUMBER'].nunique():.1f}")
    
    print(f"\nCriticality Score Range:")
    print(f"  Min: {result['criticality_score'].min():.4f}")
    print(f"  Max: {result['criticality_score'].max():.4f}")
    print(f"  Mean: {result['criticality_score'].mean():.4f}")
    
    print(f"\n5-Year Health Index Forecast:")
    print(f"  Worst 2025 HI: {result['HI_pred_2025'].min():.1f}")
    print(f"  Worst 2029 HI: {result['HI_pred_2029'].min():.1f}")
    print(f"  Average deterioration: {(result['HI_pred_2025'].mean() - result['HI_pred_2029'].mean()):.1f} HI points")
    
    print(f"\nCritical/Poor Elements (Criticality > 0.5):")
    urgent = result[result['criticality_score'] > 0.5]
    print(f"  Count: {len(urgent):,} ({len(urgent)/len(result)*100:.1f}%)")
    print(f"  Unique bridges: {urgent['STRUCTURE_NUMBER'].nunique():,}")
    
else:
    print(f"⚠ {csv_file} not found in current directory")

print("\n" + "="*60)
print("📊 DASHBOARD SETUP")
print("="*60)
print("""
File: bridge_criticality_interactive_dashboard.html
Location: same directory as element_criticality_forecast_2025_2029.csv

Steps to use:
1. Ensure CSV is exported to your working directory
2. Open browser and navigate to the HTML file
3. Select a bridge from the dropdown (or search)
4. Click "Load Dashboard" to view:
   - Metrics (criticality, HI, deterioration, CTIM impact)
   - 5 interactive charts with Plotly animations
   - Element-level detail table

Features:
✓ Bridge search/selection
✓ HI deterioration trajectory (worst vs. mean)
✓ Criticality evolution over 5 years
✓ Risk tier probabilities (pie chart)
✓ Top 5 worst elements comparison
✓ Climate-Traffic Impact heatmap
✓ Hover tooltips & zoom/pan
✓ Responsive design
✓ Dark mode support
""")


In [ ]:
# Advanced filtering utility: Find bridges by criticality, HI threshold, or deterioration rate

def find_bridges_by_criticality(df, threshold=0.30, top_n=None):
    """Find bridges where worst element exceeds criticality threshold."""
    worst_per_bridge = df.groupby("STRUCTURE_NUMBER").agg({
        "criticality_score": "max",
        "HI_pred_2025": lambda x: df[df["STRUCTURE_NUMBER"] == x.name]["HI_pred_2025"].min(),
        "HI_pred_2029": lambda x: df[df["STRUCTURE_NUMBER"] == x.name]["HI_pred_2029"].min(),
        "ctim_impact": "mean",
        "risk_tier": lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
    }).reset_index()
    
    worst_per_bridge.columns = ["STRUCTURE_NUMBER", "max_criticality", "worst_hi_2025", "worst_hi_2029", "avg_ctim", "dominant_risk"]
    worst_per_bridge["deterioration"] = worst_per_bridge["worst_hi_2025"] - worst_per_bridge["worst_hi_2029"]
    
    filtered = worst_per_bridge[worst_per_bridge["max_criticality"] >= threshold].sort_values("max_criticality", ascending=False)
    
    if top_n:
        filtered = filtered.head(top_n)
    
    return filtered

def find_bridges_by_hi_threshold(df, max_hi=50, top_n=None):
    """Find bridges where worst element HI (2029) falls below threshold."""
    worst_per_bridge = df.groupby("STRUCTURE_NUMBER").agg({
        "HI_pred_2029": lambda x: df[df["STRUCTURE_NUMBER"] == x.name]["HI_pred_2029"].min(),
        "criticality_score": "max",
        "ctim_impact": "mean"
    }).reset_index()
    
    worst_per_bridge.columns = ["STRUCTURE_NUMBER", "worst_hi_2029", "max_criticality", "avg_ctim"]
    
    filtered = worst_per_bridge[worst_per_bridge["worst_hi_2029"] <= max_hi].sort_values("worst_hi_2029")
    
    if top_n:
        filtered = filtered.head(top_n)
    
    return filtered

def find_bridges_by_deterioration(df, min_deterioration=2.0, top_n=None):
    """Find bridges with fastest deterioration rate (2025-2029)."""
    worst_per_bridge = df.groupby("STRUCTURE_NUMBER").agg({
        "HI_pred_2025": lambda x: df[df["STRUCTURE_NUMBER"] == x.name]["HI_pred_2025"].min(),
        "HI_pred_2029": lambda x: df[df["STRUCTURE_NUMBER"] == x.name]["HI_pred_2029"].min(),
        "criticality_score": "max"
    }).reset_index()
    
    worst_per_bridge.columns = ["STRUCTURE_NUMBER", "worst_hi_2025", "worst_hi_2029", "max_criticality"]
    worst_per_bridge["deterioration"] = worst_per_bridge["worst_hi_2025"] - worst_per_bridge["worst_hi_2029"]
    
    filtered = worst_per_bridge[worst_per_bridge["deterioration"] >= min_deterioration].sort_values("deterioration", ascending=False)
    
    if top_n:
        filtered = filtered.head(top_n)
    
    return filtered

# Example: Top 20 most critical bridges
print("🚨 TOP 20 MOST CRITICAL BRIDGES\n")
critical_bridges = find_bridges_by_criticality(result, threshold=0.20, top_n=20)
print(critical_bridges[["STRUCTURE_NUMBER", "max_criticality", "worst_hi_2025", "worst_hi_2029", "deterioration", "dominant_risk"]].to_string(index=False))

# Example: Bridges at high immediate risk (worst HI < 40 in 2029)
print("\n\n⚠️  BRIDGES WITH SEVERE 2029 CONDITION (HI < 40)\n")
severe_bridges = find_bridges_by_hi_threshold(result, max_hi=40, top_n=15)
if len(severe_bridges) > 0:
    print(severe_bridges[["STRUCTURE_NUMBER", "worst_hi_2029", "max_criticality"]].to_string(index=False))
else:
    print("None found in this dataset")

# Export for inspection
critical_bridges.to_csv("../results/high_priority_bridges.csv", index=False)
print("\n✓ Saved high_priority_bridges.csv")


## 10) Using the Interactive Dashboard

### Dashboard File
- **Location**: `bridge_criticality_interactive_dashboard.html` (in your project root)
- **Requires**: `element_criticality_forecast_2025_2029.csv` in the same directory
- **Open**: Double-click the HTML file or drag to browser

### Key Visualizations

#### 1. **Metrics Panel** (Top)
- **Criticality Score**: Composite rank [0–1], with risk tier badge
- **HI 2025/2029**: Worst element health indices for comparison
- **5-Year Deterioration**: Total HI point loss (worst element)
- **CTIM Impact**: Climate-Traffic impact average (shows external stressors)
- **Total Elements**: Number of monitored structural elements

#### 2. **HI Deterioration Chart**
- **Worst Element** (red, solid): Tracks the single most vulnerable element
- **Bridge Mean** (blue, dashed): Average across all elements
- Use to identify bridges with localized problems vs. portfolio-wide decline

#### 3. **Criticality Score Evolution**
- Shows how criticality changes year-to-year
- Rising trend → increasing maintenance urgency
- Use to prioritize intervention timing

#### 4. **Risk Tier Distribution** (Pie)
- Breakdown of elements by condition state (critical/poor/fair/good)
- Solid red pie slice = high risk
- Use to spot bridges with distributed vs. concentrated damage

#### 5. **Top 5 Worst Elements**
- Individual deterioration trajectories for most at-risk elements
- Identify which specific units need immediate attention
- Compare deterioration rates across elements

#### 6. **CTIM Impact Heatmap** (Climate-Traffic)
- Cell color intensity = element×hazard vulnerability
- Darker red = high climate-traffic stress on that element type
- Use to understand **why** deterioration is occurring (freeze-thaw, wet-dry, traffic abrasion, etc.)

#### 7. **Element Details Table**
- Sortable by HI, criticality, or risk tier
- Column meanings:
  - **Deterioration**: HI_2025 − HI_2029 (negative = decline)
  - **Criticality**: Composite score [0,1] (0.35+ is concerning)
  - **CTIM Impact**: Environmental stressor magnitude

---

### How to Interpret Results

**Example Bridge with HI declining from 65 → 40 (2025→2029):**
- 25-point drop in 5 years ≈ accelerated deterioration
- Check CTIM heatmap: if freeze-thaw is high, consider salt spray mitigation
- Check if traffic (ADT) is increasing: congestion may accelerate wear
- Risk tier = **poor** → maintenance windows should be planned

**Example High Criticality (0.30+):**
- Multiple elements approaching critical condition
- Network significance high (high ADT, long detour) → closure impact severe
- Action: Schedule inspection + accelerate repair bids for 2025–2026

---

### Export & Sharing

All results are saved as CSV for further analysis:

- `element_criticality_forecast_2025_2029.csv` – Full element-level data
- `bridge_criticality_summary_2025_2029.csv` – Bridge-level summaries
- `high_priority_bridges.csv` – Top candidates for immediate action

Use these to feed into your maintenance management system, asset portfolio optimization, or regulatory reporting.


### CSV Loading Issue - SOLVED ✓

If you get **"Error loading CSV file: Failed to fetch"** in the dashboard, this is a browser security (CORS) issue. Here are two solutions:

#### **Option A: Use File Upload (Easiest)**
1. Open the HTML dashboard file in your browser
2. Click **"📁 Load CSV File"** button
3. Select `element_criticality_forecast_2025_2029.csv`
4. Dashboard loads instantly!

#### **Option B: Run Local HTTP Server (Cleanest)**

**Windows**:
```powershell
cd C:\Users\clnnaLAB\Desktop\AutoDM\AutoDM
python serve_dashboard.py
```
Then open `http://localhost:8000` (opens automatically)

**Or double-click**: `Start_Dashboard_Server.bat` (one-click solution!)

**Mac/Linux**:
```bash
cd ~/Desktop/AutoDM/AutoDM
python3 -m http.server 8000
```

See **DASHBOARD_SETUP.md** for full troubleshooting guide.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                                                                                ║
║        🌉  BRIDGE CRITICALITY ANALYSIS PIPELINE - COMPLETE  ✓                 ║
║                                                                                ║
║  Temporal GCN-GRU Forecasting + Climate-Traffic Impact Matrix Integration     ║
║                                                                                ║
╚════════════════════════════════════════════════════════════════════════════════╝

📊 DELIVERABLES CREATED:

1. ✓ MODEL TRAINING & FORECASTING
   └─ Temporal GCN-GRU with state-dependent deterioration
   └─ 5-year autoregressive rollout (2025–2029)
   └─ Scenario-based covariate evolution (age, traffic, weather)

2. ✓ CRITICALITY SCORING
   └─ Climate-Traffic Impact Matrix (CTIM) computation
   └─ Composite criticality score [0,1] incorporating:
      • HI penalty (current bridge condition)
      • CTIM impact (climate + traffic stressors)
      • Network significance (ADT, detour, essential route)
   └─ Risk tier probabilities (softmax) for condition states

3. ✓ INTERACTIVE DASHBOARD
   └─ File: bridge_criticality_interactive_dashboard.html
   └─ Features:
      • Bridge search/selection dropdown
      • Real-time metrics panel (criticality, HI, deterioration)
      • HI deterioration trajectory (worst element vs. mean)
      • Criticality evolution over 5 years
      • Risk tier probability distribution (pie chart)
      • Top 5 worst elements comparison
      • Climate-Traffic Impact heatmap (element × hazard)
      • Sortable element-level detail table
      • Responsive design + dark mode support
      • Plotly.js animations & hover interactions

4. ✓ DATA EXPORTS
   └─ element_criticality_forecast_2025_2029.csv (42,879 rows)
      • Element-level HI, criticality, risk tiers, CTIM impact
   └─ bridge_criticality_summary_2025_2029.csv (5,799 rows)
      • Bridge-level aggregates (worst element per bridge)
   └─ high_priority_bridges.csv
      • Top candidates for immediate maintenance action

5. ✓ ANALYSIS UTILITIES
   └─ find_bridges_by_criticality() → Top urgent bridges
   └─ find_bridges_by_hi_threshold() → Bridges approaching critical HI
   └─ find_bridges_by_deterioration() → Fastest-declining elements

6. ✓ DOCUMENTATION
   └─ CRITICALITY_DASHBOARD_README.md
      • Architecture overview
      • Metric interpretations
      • Troubleshooting guide
      • Custom analysis examples

═══════════════════════════════════════════════════════════════════════════════

🚀 HOW TO USE:

1. EXPLORE RESULTS IN NOTEBOOK:
   └─ Run Python cells to filter by criticality, HI, or deterioration
   └─ Generate priority lists for maintenance planning
   └─ Export custom bridge subsets as CSV

2. OPEN INTERACTIVE DASHBOARD:
   └─ Open: bridge_criticality_interactive_dashboard.html in web browser
   └─ Requires: element_criticality_forecast_2025_2029.csv (same folder)
   └─ Select bridge → View all 6 panels + table
   └─ Use Plotly hover for detailed inspection
   └─ Zoom, pan, download charts as PNG

3. INTEGRATE WITH SYSTEMS:
   └─ Import CSVs into your GIS, maintenance management, or portfolio tool
   └─ Use criticality scores for budget allocation
   └─ Schedule inspections based on 2025–2026 HI predictions

═══════════════════════════════════════════════════════════════════════════════

📈 KEY INSIGHTS:

Portfolio Summary:
  • Total Bridges: 5,799
  • Total Elements: 42,879 (avg 7.4 per bridge)
  • Avg Criticality: {:.4f} / 1.0
  • Deterioration Range: {:.1f} – {:.1f} HI points / 5 years

High-Risk Subset (Criticality ≥ 0.30):
  • Bridges: {} (~{:.1f}% of portfolio)
  • Elements: {} (~{:.1f}% of elements)

Worst Forecasted HI (2029):
  • Min (single element): {:.1f}
  • Median (across all elements): {:.1f}
  • Mean: {:.1f}

═══════════════════════════════════════════════════════════════════════════════

💡 RECOMMENDED NEXT STEPS:

1. VALIDATION:
   └─ Compare 2025–2026 actual HI vs. forecast
   └─ Recalibrate scenario parameters if drift observed
   └─ Update bridge metadata (ADT, detour_km, essential routes)

2. ACTIONABLE PRIORITIZATION:
   └─ Export high_priority_bridges.csv to maintenance team
   └─ Schedule inspections for Criticality > 0.30 bridges in 2025–2026
   └─ Plan rehabilitation budgets based on HI trajectory

3. STAKEHOLDER COMMUNICATION:
   └─ Use dashboard graphics for board presentations
   └─ Export metric tables for regulatory/DOT reporting
   └─ Generate scenario analyses (e.g., "If climate warms 2°C, impact = ?")

4. CONTINUOUS IMPROVEMENT:
   └─ Retrain model annually with new NBI data
   └─ Monitor actual vs. predicted to refine delta-monotonic decay
   └─ Incorporate new hazard types (seismic, extreme weather)

═══════════════════════════════════════════════════════════════════════════════

📁 FILES IN PROJECT:

notebooks/
  └─ forecasting_arranged.ipynb ............. Main analysis (you are here)
  └─ bridge_criticality_pipeline.py ........ CTIM & criticality module

models/
  └─ temporal_gnn_bridge_hi_delta_monotonic.pt . Trained model weights

bridge_criticality_interactive_dashboard.html . Web dashboard
CRITICALITY_DASHBOARD_README.md ................. Full documentation
element_criticality_forecast_2025_2029.csv ..... Element results
bridge_criticality_summary_2025_2029.csv ....... Bridge aggregates
high_priority_bridges.csv ........................ Top candidates

═══════════════════════════════════════════════════════════════════════════════

Questions? See CRITICALITY_DASHBOARD_README.md for troubleshooting & API docs.

""".format(
    result['criticality_score'].mean(),
    (result['HI_pred_2025'] - result['HI_pred_2029']).min(),
    (result['HI_pred_2025'] - result['HI_pred_2029']).max(),
    len(result[result['criticality_score'] >= 0.30]),
    len(result[result['criticality_score'] >= 0.30]) / len(result) * 100,
    len(result[result['criticality_score'] >= 0.30]),
    len(result[result['criticality_score'] >= 0.30]) / len(result) * 100,
    result['HI_pred_2029'].min(),
    result['HI_pred_2029'].median(),
    result['HI_pred_2029'].mean()
))


## 8) Plotting
Bridge-level element trajectories + worst/median, and portfolio histogram.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

years = np.array([2025, 2026, 2027, 2028, 2029])

# pick a bridge
bridge_id = out["STRUCTURE_NUMBER"].iloc[500]

# subset of that bridge
sub = out[out["STRUCTURE_NUMBER"] == bridge_id].copy()

plt.figure(figsize=(10,6))

# plot every element trajectory
for _, r in sub.iterrows():
    traj = [r[f"HI_pred_{y}"] for y in years]
    plt.plot(years, traj, alpha=0.35, linewidth=1)

# worst element trajectory
worst = [sub[f"HI_pred_{y}"].min() for y in years]
plt.plot(years, worst, linewidth=3, marker="o", label="Worst Element")

# bridge mean trajectory
mean_vals = [sub[f"HI_pred_{y}"].mean() for y in years]
plt.plot(years, mean_vals, linewidth=3, linestyle="--", label="Bridge Mean")

plt.title(f"Element-Level Health Trajectories\nBridge {bridge_id}")
plt.xlabel("Year")
plt.ylabel("Predicted Health Index (HI)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.show()


In [ ]:
bridge_min_2029 = out.groupby("STRUCTURE_NUMBER")["HI_pred_2029"].min()

plt.figure(figsize=(9, 5))
plt.hist(bridge_min_2029, bins=40, edgecolor="black")
plt.title("Portfolio distribution of worst-element HI (2029)")
plt.xlabel("Min predicted element HI")
plt.ylabel("Number of bridges")
plt.grid(axis="y", alpha=0.3)
plt.show()
